In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from rankci import (
    rank_confidence_intervals_bootstrap,
    rank_confidence_intervals_simulation,
    rank_ci_stepwise,
    rank_ci_stepwise_pairwise,
    compute_pairwise,
    load_spf,
    load_rtdsm,
    compute_errors,
    compute_squared_error_panel,
    select_top_forecasters,
)

# Data Loading

**Forecasts**: SPF microdata (individual-level NGDP forecasts by quarter and horizon)

**Realizations**: RTDSM advance estimates (`NOUTPUTQvQd.xlsx`) — the first-published GDP figure for each quarter, avoiding contamination from later revisions.

In [ ]:
df = load_spf("../data/SPFmicrodata.xlsx")
print(f"SPF NGDP: {df.shape[0]} rows × {df.shape[1]} columns")
print("Columns:", df.columns.tolist())
df.head()

In [ ]:
noutput = load_rtdsm("../data/NOUTPUTQvQd.xlsx")
print(f"RTDSM: {noutput.shape[0]} rows × {noutput.shape[1]} columns")
noutput.head()

# Sanity Checks

In [ ]:
# Verify advance estimates for a few known quarters
from rankci.data import advance_vintage_col, get_advance_estimate

test_cases = [(1995, 2), (1999, 2), (2008, 4), (2020, 2)]
print(f"{'Target Quarter':<20} {'Vintage Col':<18} {'Value (bn $)'}")
print("-" * 55)
for y, q in test_cases:
    col = advance_vintage_col(y, q)
    val = get_advance_estimate(y, q, noutput)
    print(f"  {y}:Q{q:<16}  {col:<18}  {val:,.1f}")

# Forecast Errors

In [ ]:
# Compute all forecast errors (all horizons)
errors_df = compute_errors(df, noutput)

# Summary by horizon
print("Error summary by horizon:")
for h in range(1, 7):
    col = f"error_NGDP{h}"
    if col in errors_df.columns:
        print(f"  {col}: mean={errors_df[col].mean():+.1f}, "
              f"std={errors_df[col].std():.1f}, "
              f"nan%={errors_df[col].isna().mean()*100:.1f}%")

errors_df.head()

In [ ]:
# Squared error panel for NGDP3 (one-quarter-ahead)
# Rows = (YEAR, QUARTER), Columns = forecaster IDs, Values = squared errors
X_wide = compute_squared_error_panel(df, noutput, horizon="NGDP3")
print(f"Panel shape: {X_wide.shape[0]} quarters × {X_wide.shape[1]} forecasters")
X_wide.head()

# Forecaster Selection

In [ ]:
# Observation counts and mean MSE per forecaster
obs_counts = X_wide.notna().sum()
mean_mse   = X_wide.mean()

stats = pd.DataFrame({
    "obs_count": obs_counts,
    "mean_mse":  mean_mse,
    "rmse":      np.sqrt(mean_mse),
}).sort_values("obs_count", ascending=False)

print(f"Total forecasters: {len(stats)}")
print(f"Forecasters with >= 20 obs: {(stats['obs_count'] >= 20).sum()}")
print(f"Forecasters with >= 50 obs: {(stats['obs_count'] >= 50).sum()}")
stats.head(20)

In [ ]:
# Select top-N forecasters (adjust N here)
N = 8
X_panel = select_top_forecasters(X_wide, N=N, min_obs=20)
print(f"Selected {X_panel.shape[1]} forecasters, {X_panel.shape[0]} quarters")
print(f"Forecaster IDs: {X_panel.columns.tolist()}")

# Rank Confidence Intervals

Using the stepwise bootstrap with pairwise NW-HAC standard errors (handles the unbalanced panel).

In [ ]:
X = X_panel.values
population_ids = X_panel.columns.tolist()

out = rank_ci_stepwise_pairwise(X, alpha=0.05, B=5000, seed=42)

results = pd.DataFrame({
    "ID":         population_ids,
    "MSE":        out["theta_hat"].round(1),
    "RMSE":       np.sqrt(out["theta_hat"]).round(1),
    "CI_lower":   out["rank_ci"][:, 0],
    "CI_upper":   out["rank_ci"][:, 1],
}).sort_values("MSE")
results.index = range(1, len(results) + 1)
results.index.name = "Rank"
results